# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Initialize Dataset object
dataset = mlc.Dataset(croissant_url)

# Print metadata for overview
print("Dataset name:", dataset.metadata.name)
print("Dataset description:\n", dataset.metadata.description)

## 2. Data Overview
Review available record sets and their `@id`s, fields and columns (with their `@id`s).

We'll list the record sets, their fields, and what columns are available. This allows us to explore the data structure before extraction.

In [ ]:
# List all record sets and their IDs
record_sets_info = []
for record_set in dataset.metadata.record_sets:
    print(f"RecordSet: {record_set['@id']} - {record_set.get('name', '')}")
    field_info = []
    for field in record_set.get('fields', []):
        print(f"  Field: {field['@id']} ({field.get('name', '')})")
        if 'column' in field and field['column']:
            for col in field['column']:
                print(f"    Column: {col['@id']} ({col.get('name', '')})")
        field_info.append(field['@id'])
    record_sets_info.append(record_set['@id'])

## 3. Data Extraction
Load data from all record sets into pandas DataFrames. Each record set is referenced by its `@id`.

The following code iterates over all record sets' `@id`s found in the previous section.

In [ ]:
# Prepare to extract data from each record set
dataframes = {}

# For demonstration, list the IDs found earlier:
# record_sets_info contains a list of record set @ids

for record_set_id in record_sets_info:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"\nLoaded DataFrame for RecordSet ID: {record_set_id}")
        print("Columns:", df.columns.tolist())
        print(df.head(2))
    else:
        print(f"No records found for RecordSet {record_set_id}")

## 4. Exploratory Data Analysis (EDA)
We will select a numeric field (such as log likelihood or coefficients from the regression outputs) for EDA.

Replace `<record_set_id>` and `<numeric_field_id>` below with actual values from the previous extraction step. If available, we'll group by a categorical field such as 'ward' or 'intervention type'.

In [ ]:
# Example: Choose a record set with numeric regression results
# Replace with actual ID from data overview
example_record_set_id = record_sets_info[0] if record_sets_info else None

if example_record_set_id and example_record_set_id in dataframes:
    df = dataframes[example_record_set_id]
    print("Available columns:", df.columns.tolist())

    # Try to find a numeric field
    numeric_field = None
    for col in df.columns:
        if df[col].dtype in ['float64', 'int64']:
            numeric_field = col
            break

    if numeric_field:
        print(f"Analyzing numeric field: {numeric_field}")
        threshold = df[numeric_field].mean() if not df[numeric_field].isnull().all() else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a categorical field
        group_field = None
        for col in df.columns:
            if df[col].dtype == 'object' and col != numeric_field:
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data by {group_field}:")
            print(grouped_df.head())
    else:
        print("No numeric fields found for basic EDA.")
else:
    print("No valid record sets or DataFrames found.")

## 5. Visualization
Visualize distributions or relationships between numerical and categorical variables, such as log likelihood values or coefficients across wards or interventions.

In [ ]:
# Example visualization: Histogram and group mean
import matplotlib.pyplot as plt
import seaborn as sns

if example_record_set_id and example_record_set_id in dataframes:
    df = dataframes[example_record_set_id]
    numeric_cols = [c for c in df.columns if df[c].dtype in ['float64', 'int64']]
    cat_cols = [c for c in df.columns if df[c].dtype == 'object']

    if numeric_cols:
        field_to_plot = numeric_cols[0]
        plt.figure(figsize=(6,4))
        sns.histplot(df[field_to_plot].dropna(), kde=True)
        plt.title(f"Distribution of {field_to_plot} in {example_record_set_id}")
        plt.xlabel(field_to_plot)
        plt.ylabel('Count')
        plt.show()

    if numeric_cols and cat_cols:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=cat_cols[0], y=numeric_cols[0], data=df)
        plt.title(f"{numeric_cols[0]} by {cat_cols[0]} in {example_record_set_id}")
        plt.xlabel(cat_cols[0])
        plt.ylabel(numeric_cols[0])
        plt.show()

## 6. Conclusion
In this notebook, we explored ordered logistic regression outputs and adoption predictors from the FAIR² dataset using the `mlcroissant` library. We demonstrated how to reference entities via their `@id`, load data, analyze key numeric fields, and visualize results. This approach facilitates transparent, reproducible exploration and processing of datasets described by Croissant schemas.

Key findings include:
- Dataset offers detailed regression results affecting household adoption in rangeland management.
- Multiple numeric and categorical fields enable in-depth statistical and groupwise analysis.
- Data structure is clear via the Croissant schema, lowering the barrier to responsible, FAIR exploration.

Further steps could include modeling, more advanced visualizations, and connecting additional record sets.